# 🍷 와인 추천 RAG 챗봇

> 와인 리뷰 데이터(Kaggle 130K)를 활용하여 사용자 취향에 맞는 와인을 추천하는 RAG 챗봇

https://70b02c1cf73a722b34.gradio.live (2026년 2월 25일 (화) 오후 10시 29분까지 유효)

| 항목 | 내용 |
|---|---|
| 📦 데이터 | winemag-data-130k-v2.csv (Kaggle Wine Reviews 130K) |
| 🔤 임베딩 | paraphrase-multilingual-MiniLM-L12-v2 (384차원) |
| 🗄️ 벡터 저장소 | FAISS (IndexFlatIP) |
| 🔍 검색 | MMR (Maximal Marginal Relevance) |
| 🤖 LLM | gpt-4o-mini (OpenAI) |
| 💬 UI | Gradio Blocks (키워드 선택 + 스트리밍 챗) |
| 💱 환율 | 고정 환율 1달러 = 1,448원 |

## 🎨 기능 설계

### 🧑 페르소나
| 항목 | 내용 |
|---|---|
| 역할 | 와인 초보자를 위한 친절한 소믈리에 |
| 말투 | 친근한 존댓말 (~요체), 전문 용어는 쉽게 풀어서 설명 |
| 이름 | **무드 소믈리에** |

### 🔑 핵심 기능 — 키워드 기반 와인 추천
- UI에서 와인 타입, 가격대, 맛 프로필, 아로마, 맛, 음식, 분위기를 선택
- 선택 항목을 **텍스트로 조합**하여 RAG 체인에 전달
- 3개 와인을 마크다운 카드로 추천
- 결과 아래 채팅으로 추가 질문 가능

### 🚫 제약 조건
| 상황 | 대응 |
|---|---|
| 데이터에 정확히 맞는 와인이 없을 때 | 가장 유사한 와인을 추천하며 이유를 설명 |
| 와인과 무관한 질문 | "저는 와인 추천 전문이에요! 🍷 어떤 와인을 찾고 계신가요?" 로 유도 |

---

## 📋 구현 계획

### 1️⃣ 데이터 로드 및 전처리

**데이터 로드**
- `kagglehub.dataset_download("zynicide/wine-reviews")`로 다운로드
- pandas로 `winemag-data-130k-v2.csv` 로드 (약 130K건)

**원본 데이터 컬럼**
| 컬럼 | 설명 | 예시 |
|---|---|---|
| country | 생산 국가 | Italy, US, France |
| description | 와인 리뷰 텍스트 | "Aromas include tropical fruit..." |
| designation | 포도밭/브랜드명 | Vulkà Bianco |
| points | 평점 (80~100) | 87, 92, 95 |
| price | 가격 (USD) | 15.0, 65.0 |
| province | 생산 지역 | Sicily & Sardinia, Oregon |
| region_1, region_2 | 세부 지역 | Etna, Willamette Valley |
| variety | 포도 품종 | White Blend, Pinot Noir |
| winery | 와이너리명 | Nicosia, Rainstorm |
| title | 와인 전체 이름 | Nicosia 2013 Vulkà Bianco (Etna) |
| taster_name | 리뷰어 이름 | Kerin O'Keefe |

**필터링 (2단계)**
1. `price`가 비어있는 행 제거 → 가격 정보가 없으면 추천 시 가격 안내 불가
2. `points >= 88` → 평점 88점 이상만 남김 (일정 품질 이상의 와인만 추천하기 위해)

**랜덤 샘플링**
- 필터링 후 **3,000건** 랜덤 추출 (`sample(n=3000, random_state=42)`)

### 2️⃣ Document 객체 생성

**page_content** (임베딩 대상 텍스트)
- 와인의 핵심 정보 + 리뷰를 하나의 텍스트로 구성:
  ```
  {title}. {variety} wine from {country}, {province}.
  Priced at ${price}, rated {points}/100.
  Review: {description}
  ```
- 영어 텍스트 그대로 유지 → 다국어 임베딩 모델이 한글 질문과의 유사도 검색 처리

**청킹**
- 임베딩 모델의 최대 입력이 **128토큰**이므로, 초과 시 검색 품질 저하
- `RecursiveCharacterTextSplitter` (토큰 기준)
  - `chunk_size=120`, `chunk_overlap=20`
  - `length_function=count_tokens` (임베딩 모델 토크나이저 기준)
- 청킹 시 원본 metadata는 각 청크에 그대로 유지됨

**metadata**
| 필드 | 용도 |
|---|---|
| country | 국가 표시 |
| variety | 품종 표시 |
| price | 가격 표시 + 원화 환산 |
| points | 평점 표시 |
| winery | 와이너리 표시 |
| province | 생산 지역 표시 |
| title | 와인명 표시 |

### 3️⃣ 임베딩 + FAISS 벡터 저장소

**임베딩 모델**
| 항목 | 설정 | 이유 |
|---|---|---|
| 모델 | `paraphrase-multilingual-MiniLM-L12-v2` | 아래 참고 |
| 차원 | 384 | 모델 고정 차원 |
| device | `cpu` | MPS 미지원 모델 |

**모델 선정 사유**
- 실행 환경: Apple M1 Silicon, 16GB RAM
- `BAAI/bge-m3`(1024차원)은 모델이 커서 16GB 환경에서 부담
- `paraphrase-multilingual-MiniLM-L12-v2`(384차원)은 경량 모델로 CPU에서도 빠르게 동작
- 50개 이상 언어 지원 → 한글 질문 ↔ 영어 문서 간 검색 가능

**벡터 저장소**
| 항목 | 설정 | 이유 |
|---|---|---|
| 저장소 | FAISS | 로컬 실행, 별도 서버 불필요 |
| 인덱스 | `IndexFlatIP` | 내적 기반 검색, L2보다 연산 비용 낮음 |
| 로컬 저장 | `save_local("faiss_wine_index")` | 재실행 시 임베딩 재계산 없이 로드 |

### 4️⃣ MMR 검색기

**왜 MMR인가?**
- Top-K: 유사한 문서만 반환 → 비슷한 와인끼리 중복
- MMR: 유사도 + 다양성 고려 → 다양한 스타일의 와인 추천

**파라미터**
| 파라미터 | 값 | 의미 |
|---|---|---|
| `search_type` | `"mmr"` | MMR 검색 |
| `k` | 9 | 중복 제거 후 3개 확보를 위해 넉넉히 |
| `fetch_k` | 30 | 후보 30개에서 선별 |
| `lambda_mult` | 0.3 | 다양성 높게 (0=최대 다양성, 1=최대 유사도) |

### 5️⃣ RAG 체인 (LCEL)

**체인 구성**
```
UI 키워드 선택 → 텍스트 조합
  ↓
┌──────────────────────────────────────────┐
│ context: retriever → format_docs()       │
│ question: RunnablePassthrough()          │
└──────────────────────────────────────────┘
  ↓
ChatPromptTemplate (무드 소믈리에 프롬프트)
  ↓
ChatOpenAI (gpt-4o-mini, temperature=0)
  ↓
StrOutputParser() → 한글 와인 추천 답변
```

**키워드 → 텍스트 변환 예시**
- 선택: 레드 와인 / 3~10만원 / 산도 높음 / 장미, 베리 / 스테이크 / 데이트
- → `"레드 와인, 가격 $21~$71, 산도 높음, 장미향 베리향, 스테이크에 어울리는, 로맨틱 데이트 분위기"`

**LLM 설정**
| 항목 | 설정 | 이유 |
|---|---|---|
| 모델 | `gpt-4o-mini` | 아래 참고 |
| temperature | 0 | 일관된 추천 결과 |
| LangChain 클래스 | `ChatOpenAI` | `langchain-openai` 패키지 |

**LLM 모델 선정 사유**
- 입력 $0.15/1M, 출력 $0.60/1M → 실습 과제 수준에서 비용 부담 거의 없음
- 한글 답변 품질 우수, 지시 따르기 성능 좋음
- 응답 속도 빠름 → 스트리밍 UX에 적합

**프롬프트 요구사항**
- 역할: 와인 초보자를 위한 친절한 소믈리에 "무드 소믈리에"
- 말투: 친근한 존댓말 (~요체), 전문 용어는 쉽게 풀어서 설명
- 반드시 **한글**로 답변
- 검색된 context 기반으로만 추천 (외부 지식 사용 제한)
- 와인 **3개**를 마크다운 카드 형식으로 추천
- 각 카드에 와인명(한글 발음 + 원문 병기), 국가, 품종, 가격(달러 + 원화 환산), 점수, 추천 이유 포함
- 해외 와인이므로 **한국에서 구하기 어려울 수 있다**는 점을 안내
- 정확히 맞는 와인이 없으면 → 가장 유사한 와인을 추천하며 이유 설명
- 와인과 무관한 질문 → 와인 관련 질문으로 유도

### 6️⃣ Gradio UI

**기본 설정**
| 항목 | 설정 |
|---|---|
| 테마 | `gr.themes.Soft()` |
| 제목 | 🍷 무드 소믈리에 |
| 설명 | 오늘의 기분과 상황을 알려주세요, 딱 맞는 와인을 추천해드릴게요! |

**스트리밍**
- `chain.stream(message)`으로 청크 단위 스트리밍, `yield`로 실시간 타이핑 효과

#### 🍇 섹션 1: 와인 타입
- 버튼 그리드 (2열), **단일 선택**, 선택 시 보라색 하이라이트
- `🔴 레드 와인`, `⚪ 화이트 와인`, `🫧 스파클링 와인`, `🌸 로제 와인`, `🥃 주정강화 와인`

#### 💰 섹션 2: 가격대
- **슬라이더 2개** (최소/최대): 1만원 ~ 50만원, 기본값 3만원 ~ 10만원
- 최소 > 최대 역전 시 자동 보정
- 내부에서 고정 환율(1달러=1,448원)로 USD 변환하여 검색

#### 👅 섹션 3: 맛 프로필
- 4개 슬라이더 (1~5), 세로 배치

| 항목 | 최소 → 최대 |
|---|---|
| 🍋 산도 | 낮음 ↔ 매우 높음 |
| 🍯 당도 | 드라이 ↔ 스위트 |
| 🏋️ 바디 | 라이트 ↔ 풀바디 |
| 🫖 타닌 | 부드러움 ↔ 강함 |

- 현재 값의 한국어 레이블 실시간 표시 (예: 산도 4 → "높음")

#### 🌺 섹션 4: 아로마 키워드 (코로 맡는 향)
- **최대 5개**, 둥근 태그 버튼, **총 50개** 태그

| 카테고리 | 태그 |
|---|---|
| 🍊 감귤/시트러스 (7) | 레몬, 라임, 자몽, 오렌지, 유자, 베르가못, 레몬 제스트 |
| 🍎 과일 (8) | 사과, 배, 복숭아, 살구, 자두, 체리, 무화과, 망고 |
| 🍓 베리 (6) | 딸기, 라즈베리, 블루베리, 블랙베리, 크랜베리, 블랙커런트 |
| 🌸 꽃 (7) | 장미, 라벤더, 자스민, 바이올렛, 흰 꽃, 아카시아꽃, 허니서클 |
| 🌿 허브 (5) | 민트, 로즈마리, 타임, 세이지, 허브 |
| 🫚 스파이스 (5) | 시나몬, 후추, 정향, 바닐라, 생강 |
| 🪵 우드/어스 (5) | 오크, 삼나무, 가죽, 흙, 미네랄 |
| 🔥 로스팅/너티 (7) | 커피, 코코아, 토스트, 아몬드, 헤이즐넛, 스모키, 구운 빵 |

#### 😋 섹션 5: 맛 키워드 (입에서 느끼는 맛과 질감)
- **최대 5개**, **총 50개** 태그

| 카테고리 | 태그 |
|---|---|
| 💧 질감/바디 (7) | 가벼운, 부드러운, 크리미, 실키, 묵직한, 벨벳, 탄탄한 |
| 🍬 당도/단맛 (5) | 달콤한, 드라이, 은은한 단맛, 꿀맛, 캐러멜맛 |
| 🍋 산미/신맛 (5) | 상큼한, 산뜻한, 새콤한, 청량한, 상쾌한 |
| 🫖 타닌/쓴맛 (4) | 쌉쌀한, 부드러운 타닌, 단단한 구조감, 떫은 |
| 🫧 탄산/청량 (3) | 탄산감, 경쾌한 탄산, 크리미한 거품 |
| 🎵 밸런스/인상 (8) | 균형 잡힌, 복합적인, 조화로운, 우아한, 섬세한, 풍성한, 진한, 깊은 |
| 🍇 과일맛 (7) | 잘 익은 과일, 과일 잼, 말린 과일, 농축된 과일, 잘 익은 베리, 달콤한 과일, 상큼한 베리 |
| 🏁 여운/피니시 (5) | 긴 여운, 깔끔한, 풍부한, 프루티, 스파이시 |
| ✨ 전체 느낌 (6) | 화사한, 생동감, 따뜻한, 신선한, 쥬시, 강렬한 |

#### 🍽️ 섹션 6: 음식 선택
- **최대 3개**

| 카테고리 | 태그 |
|---|---|
| 🥩 육류 | 스테이크, 양갈비, 삼겹살, 닭요리, 오리 |
| 🐟 해산물 | 회/초밥, 조개/굴, 새우, 연어, 랍스터 |
| 🧀 치즈/전채 | 치즈 플래터, 브루스케타, 올리브, 샐러드 |
| 🍝 파스타/피자 | 크림 파스타, 토마토 파스타, 피자 |
| 🇰🇷 한식 | 불고기, 갈비찜, 김치찜, 해물탕 |
| 🍰 디저트 | 초콜릿 케이크, 과일, 마카롱, 치즈케이크 |

#### 🌙 섹션 7: 분위기
- **단일 선택**
- `💕 로맨틱 데이트`, `🎉 친구 모임`, `💼 비즈니스 미팅`, `🥂 혼술`, `👨‍👩‍👧‍👦 가족 식사`, `🎊 축하/파티`, `😎 캐주얼`, `🎩 격식 있는 자리`, `🏕️ 야외/피크닉`

#### 🔍 섹션 8: 검색 버튼
- **"🔍 와인 검색"** — 진한 보라색 배경, 흰색 텍스트, 전체 너비, 둥근 모서리

#### 🍷 섹션 9: 결과 영역
- 검색 버튼 클릭 후 카드 형태로 추천 와인 표시
- 결과 아래에 **채팅 인터페이스** 배치 (추가 질문 가능)

### 📌 레퍼런스 서비스
- [WARVIS](https://warvis.kr/home) — 와인 타입, 가격대, 맛 프로필, 아로마 키워드로 와인을 추천하는 서비스
- 이 서비스의 키워드 기반 추천 방식을 챗봇 형태로 구현

![레퍼런스 서비스](./2주차%20실습과제%20레퍼런스%20서비스.png)

## 🔧 구현 후 수정 과정

### 1. Import 경로 수정

| 변경 전 | 변경 후 | 사유 |
|---|---|---|
| `from langchain.schema import Document` | `from langchain_core.documents import Document` | `langchain.schema` 모듈이 최신 버전에서 제거됨 |

### 2. Gradio 6.x 호환성 대응

| 항목 | 변경 내용 | 사유 |
|---|---|---|
| `gr.Blocks()` 파라미터 | `theme`, `css`를 `launch()`로 이동 | Gradio 6.0에서 Blocks 생성자에서 launch()로 파라미터 위치 변경 |
| `gr.Chatbot(type="messages")` | `type` 파라미터 제거 | Gradio 6.x에서 `type` 파라미터 미지원 (messages가 기본값) |
| `gr.Slider(value=[30000, 100000])` | `price_min`, `price_max` 슬라이더 2개로 분리 | Gradio 6.x에서 이중 슬라이더(range) 미지원, `int` 반환으로 언패킹 에러 발생 |

### 3. LLM 모델 변경

| 변경 전 | 변경 후 | 사유 |
|---|---|---|
| `gemini-2.0-flash` (Google) | `gpt-4o-mini` (OpenAI) | Gemini 무료 티어 rate limit 초과 (429 RESOURCE_EXHAUSTED), quota limit: 0 |
| `ChatGoogleGenerativeAI` | `ChatOpenAI` | LLM 클래스 변경에 따른 import 및 호출부 수정 |

### 4. 청킹에 의한 와인 중복 문제 해결

**문제**: `RecursiveCharacterTextSplitter`로 청킹 시 하나의 와인이 여러 청크로 분할됨 (3,000건 → 3,930개 벡터). MMR 검색 결과 같은 와인의 서로 다른 청크가 반환되어 중복 추천 발생.

**해결**:

| 항목 | 변경 전 | 변경 후 |
|---|---|---|
| retriever `k` | 3 | 9 (중복 제거 후에도 3개 와인 확보) |
| retriever `fetch_k` | 10 | 30 |
| `format_docs()` | 청크를 그대로 나열 | title 기준 중복 제거 + 같은 와인 리뷰 텍스트 병합 후 상위 3개만 사용 |

### 5. 환율 업데이트

| 변경 전 | 변경 후 |
|---|---|
| `EXCHANGE_RATE = 1400` | `EXCHANGE_RATE = 1448` |

### 6. 와인명 한글 발음 표기

**변경**: 시스템 프롬프트에 와인명을 한글 발음으로 변환하고 원문을 괄호 병기하도록 지시 추가.

- 예: `Nicosia 2013 Vulkà Bianco` → `니코시아 2013 불카 비앙코 (Nicosia 2013 Vulkà Bianco)`
- 와인 초보자가 영문명만으로는 읽기 어려운 점을 고려

### 7. 가격 슬라이더 역전 방지

**문제**: 최소 가격 슬라이더를 최대 가격보다 높게 설정할 수 있어, 가격 범위가 역전된 채로 검색됨.

**해결**:
- **UI**: `price_min.change()` / `price_max.change()` 이벤트로 슬라이더 연동. 최소를 올리면 최대가 따라가고, 최대를 내리면 최소가 따라 내려감.
- **백엔드**: `compose_query()`에서 `sorted()`로 한 번 더 보정하여 안전하게 처리.

In [2]:
"""
🍷 무드 소믈리에 — 와인 추천 RAG 챗봇
와인 리뷰 데이터(Kaggle 130K)를 활용하여 사용자 취향에 맞는 와인을 추천하는 RAG 챗봇
"""

import pathlib

import gradio as gr
import kagglehub
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

load_dotenv()

# ──────────────────────────────────────────────
# 경로 설정
# ──────────────────────────────────────────────
try:
    BASE_DIR = pathlib.Path(__file__).resolve().parent
except NameError:
    # Jupyter notebook 환경: 노트북 파일 기준 경로
    BASE_DIR = pathlib.Path.cwd()

# kagglehub로 데이터 다운로드
DATA_DIR = pathlib.Path(kagglehub.dataset_download("zynicide/wine-reviews"))
DATA_PATH = DATA_DIR / "winemag-data-130k-v2.csv"
FAISS_DIR = BASE_DIR / "faiss_wine_index"

# ──────────────────────────────────────────────
# 상수
# ──────────────────────────────────────────────
EXCHANGE_RATE = 1448  # 1 USD = 1,448 KRW
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
LLM_MODEL = "gpt-4o-mini"
SAMPLE_N = 3000
RANDOM_STATE = 42

# ──────────────────────────────────────────────
# 1️⃣ 데이터 로드 및 전처리
# ──────────────────────────────────────────────

def load_and_preprocess() -> pd.DataFrame:
    """CSV 로드 → 필터링 → 3,000건 랜덤 샘플링."""
    df = pd.read_csv(DATA_PATH, index_col=0)
    df = df[df["price"].notna()]
    df = df[df["points"] >= 88]
    df = df.sample(n=SAMPLE_N, random_state=RANDOM_STATE)
    return df.reset_index(drop=True)


# ──────────────────────────────────────────────
# 2️⃣ Document 객체 생성 + 청킹
# ──────────────────────────────────────────────

def _build_page_content(row: pd.Series) -> str:
    return (
        f"{row['title']}. {row['variety']} wine from {row['country']}, {row['province']}. "
        f"Priced at ${row['price']}, rated {row['points']}/100. "
        f"Review: {row['description']}"
    )


def _build_metadata(row: pd.Series) -> dict:
    return {
        "country": row.get("country", ""),
        "variety": row.get("variety", ""),
        "price": float(row["price"]),
        "points": int(row["points"]),
        "winery": row.get("winery", ""),
        "province": row.get("province", ""),
        "title": row.get("title", ""),
    }


def create_documents(df: pd.DataFrame) -> list[Document]:
    """DataFrame → Document 리스트 (청킹 포함)."""
    tokenizer = AutoTokenizer.from_pretrained(f"sentence-transformers/{EMBEDDING_MODEL}")

    def count_tokens(text: str) -> int:
        return len(tokenizer.encode(text, add_special_tokens=False))

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=120,
        chunk_overlap=20,
        length_function=count_tokens,
    )

    docs: list[Document] = []
    for _, row in df.iterrows():
        content = _build_page_content(row)
        metadata = _build_metadata(row)
        chunks = splitter.split_text(content)
        for chunk in chunks:
            docs.append(Document(page_content=chunk, metadata=metadata))

    print(f"✅ Document 생성 완료: {len(docs)}개 (원본 {len(df)}건)")
    return docs


# ──────────────────────────────────────────────
# 3️⃣ 임베딩 + FAISS 벡터 저장소
# ──────────────────────────────────────────────

def get_embedding_model() -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},
    )


def build_or_load_vectorstore(embedding_model: HuggingFaceEmbeddings) -> FAISS:
    """FAISS 인덱스가 있으면 로드, 없으면 생성 후 저장."""
    if FAISS_DIR.exists():
        print("📂 기존 FAISS 인덱스 로드 중...")
        vs = FAISS.load_local(
            str(FAISS_DIR), embedding_model, allow_dangerous_deserialization=True
        )
        print(f"✅ FAISS 로드 완료: {vs.index.ntotal}개 벡터")
        return vs

    print("🔨 FAISS 인덱스 새로 생성 중 (최초 1회)...")
    df = load_and_preprocess()
    docs = create_documents(df)
    vs = FAISS.from_documents(docs, embedding_model)
    vs.save_local(str(FAISS_DIR))
    print(f"✅ FAISS 저장 완료: {vs.index.ntotal}개 벡터 → {FAISS_DIR}")
    return vs


# ──────────────────────────────────────────────
# 4️⃣ MMR 검색기
# ──────────────────────────────────────────────

def get_retriever(vectorstore: FAISS):
    # 청킹으로 같은 와인의 청크가 여러 개 존재하므로,
    # 중복 제거 후에도 3개 와인을 확보하기 위해 k=9로 넉넉히 가져옴
    return vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 9, "fetch_k": 30, "lambda_mult": 0.3},
    )


# ──────────────────────────────────────────────
# 5️⃣ RAG 체인 (LCEL)
# ──────────────────────────────────────────────

SYSTEM_PROMPT = """\
당신은 **무드 소믈리에**입니다.
와인 초보자를 위한 친절한 소믈리에로, 친근한 존댓말(~요체)을 사용하고 전문 용어는 쉽게 풀어서 설명해주세요.

## 규칙
1. 반드시 **한글**로 답변하세요.
2. 아래 [검색된 와인 정보]만 활용하여 추천하세요. 외부 지식은 사용하지 마세요.
3. **3개** 와인을 마크다운 카드 형식으로 추천하세요.
4. 해외 와인이므로 **한국에서 구하기 어려울 수 있다**는 점을 안내하세요.
5. 정확히 맞는 와인이 없으면 가장 유사한 와인을 추천하며 이유를 설명하세요.
6. 와인과 무관한 질문에는 "저는 와인 추천 전문이에요! 🍷 어떤 와인을 찾고 계신가요?"로 유도하세요.
7. 환율은 1달러 = 1,400원으로 환산하세요.
8. 와인명은 **한글 발음 표기**로 변환하고, 괄호 안에 원문을 병기하세요. 예: 니코시아 2013 불카 비앙코 (Nicosia 2013 Vulkà Bianco)

## 카드 형식
각 와인마다 아래 형식을 사용하세요:

### 🥇 1. 한글 발음 와인명 (원문 와인명)
- 🍇 **품종**: [품종] | 🌍 **국가/지역**: [국가], [지역]
- 💰 **가격**: $[USD] (약 ₩[원화]) | ⭐ **평점**: [점수]/100
- 💡 **추천 이유**: [사용자 요청에 맞춰 설명]

---

[검색된 와인 정보]
{context}
"""

USER_PROMPT = "{question}"


def format_docs(docs: list[Document]) -> str:
    """검색된 Document 청크를 LLM 컨텍스트용 텍스트로 변환.

    같은 와인의 청크가 여러 개 반환될 수 있으므로,
    title 기준으로 중복을 제거하고 최대 3개 와인만 사용한다.
    같은 와인의 청크가 여러 개면 리뷰 텍스트를 합쳐 정보 손실을 방지한다.
    """
    # title 기준으로 중복 제거 + 리뷰 텍스트 병합
    seen: dict[str, dict] = {}  # {title: {metadata, reviews: [...]}}
    for doc in docs:
        title = doc.metadata.get("title", "")
        if title not in seen:
            seen[title] = {"metadata": doc.metadata, "reviews": []}
        seen[title]["reviews"].append(doc.page_content)

    # 상위 3개 와인만 사용 (retriever 반환 순서 = 유사도 순)
    formatted = []
    for i, (title, info) in enumerate(list(seen.items())[:3], 1):
        m = info["metadata"]
        review = " ".join(info["reviews"])
        krw = int(m["price"] * EXCHANGE_RATE)
        formatted.append(
            f"[와인 {i}]\n"
            f"- 와인명: {m['title']}\n"
            f"- 품종: {m['variety']}\n"
            f"- 국가: {m['country']}, {m['province']}\n"
            f"- 와이너리: {m['winery']}\n"
            f"- 가격: ${m['price']} (약 ₩{krw:,})\n"
            f"- 평점: {m['points']}/100\n"
            f"- 리뷰: {review}\n"
        )
    return "\n".join(formatted)


def build_chain(retriever):
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", USER_PROMPT),
    ])
    llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


# ──────────────────────────────────────────────
# 6️⃣ Gradio UI — 키워드 선택 + 스트리밍 챗
# ──────────────────────────────────────────────

# ── 키워드 데이터 ──

WINE_TYPES = ["🔴 레드 와인", "⚪ 화이트 와인", "🫧 스파클링 와인", "🌸 로제 와인", "🥃 주정강화 와인"]

TASTE_LABELS = {
    "acidity": {1: "낮음", 2: "약간 낮음", 3: "보통", 4: "높음", 5: "매우 높음"},
    "sweetness": {1: "드라이", 2: "약간 드라이", 3: "보통", 4: "약간 스위트", 5: "스위트"},
    "body": {1: "라이트", 2: "미디엄 라이트", 3: "미디엄", 4: "미디엄 풀", 5: "풀바디"},
    "tannin": {1: "부드러움", 2: "약간 부드러움", 3: "보통", 4: "약간 강함", 5: "강함"},
}

AROMA_TAGS = {
    "🍊 감귤/시트러스": ["레몬", "라임", "자몽", "오렌지", "유자", "베르가못", "레몬 제스트"],
    "🍎 과일": ["사과", "배", "복숭아", "살구", "자두", "체리", "무화과", "망고"],
    "🍓 베리": ["딸기", "라즈베리", "블루베리", "블랙베리", "크랜베리", "블랙커런트"],
    "🌸 꽃": ["장미", "라벤더", "자스민", "바이올렛", "흰 꽃", "아카시아꽃", "허니서클"],
    "🌿 허브": ["민트", "로즈마리", "타임", "세이지", "허브"],
    "🫚 스파이스": ["시나몬", "후추", "정향", "바닐라", "생강"],
    "🪵 우드/어스": ["오크", "삼나무", "가죽", "흙", "미네랄"],
    "🔥 로스팅/너티": ["커피", "코코아", "토스트", "아몬드", "헤이즐넛", "스모키", "구운 빵"],
}

TASTE_TAGS = {
    "💧 질감/바디": ["가벼운", "부드러운", "크리미", "실키", "묵직한", "벨벳", "탄탄한"],
    "🍬 당도/단맛": ["달콤한", "드라이", "은은한 단맛", "꿀맛", "캐러멜맛"],
    "🍋 산미/신맛": ["상큼한", "산뜻한", "새콤한", "청량한", "상쾌한"],
    "🫖 타닌/쓴맛": ["쌉쌀한", "부드러운 타닌", "단단한 구조감", "떫은"],
    "🫧 탄산/청량": ["탄산감", "경쾌한 탄산", "크리미한 거품"],
    "🎵 밸런스/인상": ["균형 잡힌", "복합적인", "조화로운", "우아한", "섬세한", "풍성한", "진한", "깊은"],
    "🍇 과일맛": ["잘 익은 과일", "과일 잼", "말린 과일", "농축된 과일", "잘 익은 베리", "달콤한 과일", "상큼한 베리"],
    "🏁 여운/피니시": ["긴 여운", "깔끔한", "풍부한", "프루티", "스파이시"],
    "✨ 전체 느낌": ["화사한", "생동감", "따뜻한", "신선한", "쥬시", "강렬한"],
}

FOOD_TAGS = {
    "🥩 육류": ["스테이크", "양갈비", "삼겹살", "닭요리", "오리"],
    "🐟 해산물": ["회/초밥", "조개/굴", "새우", "연어", "랍스터"],
    "🧀 치즈/전채": ["치즈 플래터", "브루스케타", "올리브", "샐러드"],
    "🍝 파스타/피자": ["크림 파스타", "토마토 파스타", "피자"],
    "🇰🇷 한식": ["불고기", "갈비찜", "김치찜", "해물탕"],
    "🍰 디저트": ["초콜릿 케이크", "과일", "마카롱", "치즈케이크"],
}

MOOD_OPTIONS = [
    "💕 로맨틱 데이트", "🎉 친구 모임", "💼 비즈니스 미팅",
    "🥂 혼술", "👨‍👩‍👧‍👦 가족 식사", "🎊 축하/파티",
    "😎 캐주얼", "🎩 격식 있는 자리", "🏕️ 야외/피크닉",
]


# ── 키워드 → 검색 텍스트 변환 ──

def compose_query(
    wine_type: str,
    price_min: int,
    price_max: int,
    acidity: int,
    sweetness: int,
    body: int,
    tannin: int,
    aromas: list[str],
    tastes: list[str],
    foods: list[str],
    mood: str,
) -> str:
    """UI 선택 항목을 RAG 검색용 텍스트로 조합."""
    parts: list[str] = []

    if wine_type:
        parts.append(wine_type.split(" ", 1)[-1])  # 이모지 제거

    # 최소 > 최대인 경우 자동 보정
    lo, hi = sorted([price_min, price_max])
    min_usd = round(lo / EXCHANGE_RATE)
    max_usd = round(hi / EXCHANGE_RATE)
    parts.append(f"가격 ${min_usd}~${max_usd}")

    parts.append(f"산도 {TASTE_LABELS['acidity'][acidity]}")
    parts.append(f"당도 {TASTE_LABELS['sweetness'][sweetness]}")
    parts.append(f"바디 {TASTE_LABELS['body'][body]}")
    parts.append(f"타닌 {TASTE_LABELS['tannin'][tannin]}")

    if aromas:
        parts.append(" ".join(f"{a}향" for a in aromas))
    if tastes:
        parts.append(" ".join(tastes))
    if foods:
        parts.append(" ".join(f"{f}에 어울리는" for f in foods))
    if mood:
        parts.append(mood.split(" ", 1)[-1] + " 분위기")

    return ", ".join(parts)


# ── CSS ──

CUSTOM_CSS = """
/* 와인 타입 / 분위기 라디오 버튼 */
.wine-type-radio .wrap,
.mood-radio .wrap {
    display: grid !important;
    grid-template-columns: repeat(2, 1fr) !important;
    gap: 8px !important;
}
.wine-type-radio label,
.mood-radio label {
    border: 2px solid #e0d4f5 !important;
    border-radius: 12px !important;
    padding: 10px 14px !important;
    text-align: center !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
}
.wine-type-radio input:checked + label,
.mood-radio input:checked + label {
    border-color: #7c3aed !important;
    background: #f3e8ff !important;
}

/* 태그 체크박스 (아로마, 맛, 음식) */
.tag-group .wrap {
    display: flex !important;
    flex-wrap: wrap !important;
    gap: 6px !important;
}
.tag-group label {
    border: 1.5px solid #e0d4f5 !important;
    border-radius: 999px !important;
    padding: 6px 14px !important;
    font-size: 0.85rem !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
}
.tag-group input:checked + label {
    border-color: #7c3aed !important;
    background: #f3e8ff !important;
    color: #5b21b6 !important;
}

/* 검색 버튼 */
.search-btn {
    background: #6d28d9 !important;
    color: white !important;
    border-radius: 12px !important;
    font-size: 1.1rem !important;
    padding: 12px !important;
}
.search-btn:hover {
    background: #5b21b6 !important;
}

/* 슬라이더 라벨 */
.slider-label {
    font-size: 0.85rem;
    color: #6b7280;
    text-align: center;
    margin-top: -4px;
}
"""


def create_ui(chain):
    """Gradio Blocks UI 생성."""

    # ── 선택 제한 헬퍼 ──

    def limit_selection(selected: list[str], max_n: int) -> list[str]:
        """최대 개수 초과 시 마지막 선택 제거."""
        if len(selected) > max_n:
            return selected[:max_n]
        return selected

    # ── 맛 프로필 라벨 업데이트 ──

    def acidity_label(v):
        return f"현재: {TASTE_LABELS['acidity'][v]}"

    def sweetness_label(v):
        return f"현재: {TASTE_LABELS['sweetness'][v]}"

    def body_label(v):
        return f"현재: {TASTE_LABELS['body'][v]}"

    def tannin_label(v):
        return f"현재: {TASTE_LABELS['tannin'][v]}"

    # ── 검색 함수 ──

    def search_wine(
        wine_type, price_min, price_max, acidity, sweetness, body, tannin,
        aromas, tastes, foods, mood, history,
    ):
        query = compose_query(
            wine_type, price_min, price_max, acidity, sweetness, body, tannin,
            aromas, tastes, foods, mood,
        )
        history = history or []
        history.append({"role": "user", "content": f"🔍 검색 조건: {query}"})
        response = ""
        for chunk in chain.stream(query):
            response += chunk
            yield history + [{"role": "assistant", "content": response}]

    # ── 채팅 함수 ──

    def chat_fn(message: str, history: list[dict]):
        if not message.strip():
            yield history
            return
        history = history or []
        history.append({"role": "user", "content": message})
        response = ""
        for chunk in chain.stream(message):
            response += chunk
            yield history + [{"role": "assistant", "content": response}]

    # ── UI 구성 ──

    with gr.Blocks(title="🍷 무드 소믈리에") as demo:
        gr.Markdown("# 🍷 무드 소믈리에\n> 오늘의 기분과 상황을 알려주세요, 딱 맞는 와인을 추천해드릴게요!")

        with gr.Row():
            # ── 왼쪽: 키워드 선택 패널 ──
            with gr.Column(scale=1):
                # 섹션 1: 와인 타입
                gr.Markdown("### 🍇 와인 타입")
                wine_type = gr.Radio(
                    choices=WINE_TYPES,
                    value="🔴 레드 와인",
                    label="",
                    elem_classes=["wine-type-radio"],
                )

                # 섹션 2: 가격대
                gr.Markdown("### 💰 가격대")
                price_min = gr.Slider(
                    minimum=10000, maximum=500000,
                    value=30000, step=10000,
                    label="최소 가격 (원)",
                )
                price_max = gr.Slider(
                    minimum=10000, maximum=500000,
                    value=100000, step=10000,
                    label="최대 가격 (원)",
                )
                # 최소 > 최대 역전 방지: 슬라이더 변경 시 상대 슬라이더 자동 보정
                price_min.change(
                    lambda lo, hi: max(lo, hi), [price_min, price_max], price_max,
                )
                price_max.change(
                    lambda hi, lo: min(hi, lo), [price_max, price_min], price_min,
                )

                # 섹션 3: 맛 프로필
                gr.Markdown("### 👅 맛 프로필")

                acidity = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="🍋 산도")
                acidity_lbl = gr.Markdown("현재: 보통", elem_classes=["slider-label"])
                acidity.change(acidity_label, acidity, acidity_lbl)

                sweetness = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="🍯 당도")
                sweetness_lbl = gr.Markdown("현재: 보통", elem_classes=["slider-label"])
                sweetness.change(sweetness_label, sweetness, sweetness_lbl)

                body = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="🏋️ 바디")
                body_lbl = gr.Markdown("현재: 미디엄", elem_classes=["slider-label"])
                body.change(body_label, body, body_lbl)

                tannin = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="🫖 타닌")
                tannin_lbl = gr.Markdown("현재: 보통", elem_classes=["slider-label"])
                tannin.change(tannin_label, tannin, tannin_lbl)

                # 섹션 4: 아로마
                gr.Markdown("### 🌺 아로마 키워드 (최대 5개)")
                all_aromas = [tag for tags in AROMA_TAGS.values() for tag in tags]
                aromas = gr.CheckboxGroup(
                    choices=all_aromas, value=[], label="", elem_classes=["tag-group"],
                )
                aromas.change(
                    lambda s: limit_selection(s, 5), aromas, aromas,
                )

                # 섹션 5: 맛 키워드
                gr.Markdown("### 😋 맛 키워드 (최대 5개)")
                all_tastes = [tag for tags in TASTE_TAGS.values() for tag in tags]
                tastes = gr.CheckboxGroup(
                    choices=all_tastes, value=[], label="", elem_classes=["tag-group"],
                )
                tastes.change(
                    lambda s: limit_selection(s, 5), tastes, tastes,
                )

                # 섹션 6: 음식
                gr.Markdown("### 🍽️ 음식 (최대 3개)")
                all_foods = [tag for tags in FOOD_TAGS.values() for tag in tags]
                foods = gr.CheckboxGroup(
                    choices=all_foods, value=[], label="", elem_classes=["tag-group"],
                )
                foods.change(
                    lambda s: limit_selection(s, 3), foods, foods,
                )

                # 섹션 7: 분위기
                gr.Markdown("### 🌙 분위기")
                mood = gr.Radio(
                    choices=MOOD_OPTIONS,
                    value="😎 캐주얼",
                    label="",
                    elem_classes=["mood-radio"],
                )

                # 섹션 8: 검색 버튼
                search_btn = gr.Button(
                    "🔍 와인 검색", variant="primary", elem_classes=["search-btn"],
                )

            # ── 오른쪽: 결과 + 채팅 ──
            with gr.Column(scale=1):
                # 섹션 9: 결과 및 채팅
                gr.Markdown("### 🍷 추천 결과")
                chatbot = gr.Chatbot(
                    height=600,
                    placeholder="왼쪽에서 취향을 선택하고 '와인 검색' 버튼을 눌러보세요!",
                )

                with gr.Row():
                    chat_input = gr.Textbox(
                        placeholder="추가 질문을 입력하세요...",
                        show_label=False,
                        scale=4,
                    )
                    send_btn = gr.Button("전송", scale=1)

        # ── 이벤트 연결 ──
        search_btn.click(
            fn=search_wine,
            inputs=[
                wine_type, price_min, price_max, acidity, sweetness, body, tannin,
                aromas, tastes, foods, mood, chatbot,
            ],
            outputs=[chatbot],
        )

        send_btn.click(
            fn=chat_fn,
            inputs=[chat_input, chatbot],
            outputs=[chatbot],
        ).then(lambda: "", outputs=[chat_input])

        chat_input.submit(
            fn=chat_fn,
            inputs=[chat_input, chatbot],
            outputs=[chatbot],
        ).then(lambda: "", outputs=[chat_input])

    return demo


# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────

def main():
    print("🍷 무드 소믈리에 시작...")
    embedding_model = get_embedding_model()
    vectorstore = build_or_load_vectorstore(embedding_model)
    retriever = get_retriever(vectorstore)
    chain = build_chain(retriever)
    demo = create_ui(chain)
    demo.launch(theme=gr.themes.Soft(), css=CUSTOM_CSS, share=True)


if __name__ == "__main__":
    main()

🍷 무드 소믈리에 시작...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2003.10it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📂 기존 FAISS 인덱스 로드 중...
✅ FAISS 로드 완료: 3930개 벡터
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://9cfef665ec609df0e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
